In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import time

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@stgdatabricksete14.dfs.core.windows.net/products")
display(df)

In [0]:
df = df.drop("_rescued_data")
display(df)

In [0]:
df.createOrReplaceTempView("products_view")

**Functions**

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_catalog.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.90


In [0]:
%sql
select product_id,price, databricks_catalog.bronze.discount_func(price) as discounted_price FROM products_view

In [0]:
df = df.withColumn("discounted_price",expr('databricks_catalog.bronze.discount_func(price)'))
df.display()

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_catalog.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
   return p_brand.upper()
$$

In [0]:
%sql
select product_id, brand, databricks_catalog.bronze.upper_func(brand) as brand_upper FROM products_view

In [0]:
df.write.format('delta').mode('overwrite').option('path','abfss://silver@stgdatabricksete14.dfs.core.windows.net/products').save()

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_catalog.silver.products_silver
using DELTA
LOCATION "abfss://silver@stgdatabricksete14.dfs.core.windows.net/products"